# StarDist 2D セグメンテーション notebook

この notebook は、Google Drive に保存した **2D TIFF 画像**を StarDist の学習済みモデル `2D_versatile_fluo` でインスタンスセグメンテーションするためのものです。

- 対象: 蛍光画像の核など、単一チャンネルの 2D intensity 画像
- 入力: `.tif` または `.tiff`
- 出力: ラベル画像 TIFF、オーバーレイ PNG、簡単な測定 CSV
- 使用モデル: `StarDist2D.from_pretrained("2D_versatile_fluo")`
- 実行前設定: Colab の `ランタイム` > `ランタイムのタイプを変更` で、ハードウェア アクセラレータに `GPU` を選択してください。

注意: `2D_versatile_fluo` は 2D 蛍光核画像向けの学習済みモデルです。H&E などの RGB brightfield 画像では `2D_versatile_he` の方が適切です。3D TIFF にはこの notebook は使いません。

## 1. 必要なパッケージのインストール

初回実行後に Colab で「セッションがクラッシュしました」と表示されることがあります。依存関係を反映するための再起動なので、再接続されたら次のセルに進んでください。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

install_marker = Path("/content/.stardist_dependencies_installed")
if not install_marker.exists():
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--force-reinstall",
        "numpy==1.26.4",
        "numba>=0.60,<0.62",
        "stardist",
        "tifffile",
        "imagecodecs",
        "pandas",
    ])
    install_marker.write_text("done")
    os.kill(os.getpid(), 9)

## 2. Google Drive をマウント

入力 TIFF と出力先フォルダにアクセスするため、Google Drive を Colab に接続します。表示される認証手順に従ってください。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. ライブラリの読み込み

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from IPython.display import Markdown, display
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from stardist.plot import render_label

plt.rcParams["figure.figsize"] = (7, 7)

## 4. 入力画像と出力先を設定

`input_tiff_path` を Google Drive 上の 2D TIFF 画像に変更してください。

multi-channel TIFF の場合は `use_channel_selection` を `True` にして、`channel_index` で使うチャンネルを指定します。ただし、`2D_versatile_fluo` は単一チャンネル蛍光画像向けです。

| パラメータ | 意味 |
|---|---|
| `input_tiff_path` | 入力する 2D TIFF 画像のパス |
| `output_dir` | 結果を保存するフォルダ |
| `use_channel_selection` | multi-channel 画像から 1 チャンネルだけ使う場合は `True` |
| `channel_index` | 使用するチャンネル番号。0 始まり |
| `percentile_low` | 正規化で下限として使う percentile |
| `percentile_high` | 正規化で上限として使う percentile |
| `use_custom_thresholds` | `True` にすると `prob_thresh` / `nms_thresh` を使う |
| `prob_thresh` | 検出の信頼度しきい値。低いほど検出が増えやすい (default: 0.5)|
| `nms_thresh` | 重なった候補を統合するしきい値。低いほど物体がくっつきやすい (default: 0.3)|
| `n_tiles_y` | Y 方向のタイル分割数。メモリ不足時に増やす |
| `n_tiles_x` | X 方向のタイル分割数。メモリ不足時に増やす |

In [ ]:
input_tiff_path = "/content/drive/MyDrive/path/to/input_image.tif"  #@param {type:"string"}
output_dir = "/content/drive/MyDrive/stardist_2d_results"  #@param {type:"string"}

use_channel_selection = False  #@param {type:"boolean"}
channel_index = 0  #@param {type:"integer"}

percentile_low = 1.0  #@param {type:"number"}
percentile_high = 99.8  #@param {type:"number"}

use_custom_thresholds = False  #@param {type:"boolean"}
prob_thresh = 0.5  #@param {type:"number"}
nms_thresh = 0.3  #@param {type:"number"}
n_tiles_y = 1  #@param {type:"integer"}
n_tiles_x = 1  #@param {type:"integer"}

input_tiff_path = Path(input_tiff_path)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

## 5. TIFF 画像を読み込み、形式とビット深度を確認

画像の `shape`, `dtype`, 値域を確認します。

想定する入力は単一チャンネルの 2D intensity 画像です。主な想定 dtype は `uint8`, `uint16`, `float32`, `float64` です。StarDist へ渡す前に intensity 正規化を行うため、8-bit と 16-bit の違い自体は通常問題になりません。ただし、画像が飽和している、背景が強い、対象物サイズが学習データと大きく異なる場合は結果が悪くなることがあります。

RGB/RGBA や `C, Y, X` 形式の multi-channel TIFF は、そのままでは `2D_versatile_fluo` の入力として扱いません。必要なチャンネルを選択してください。

In [ ]:
if not input_tiff_path.exists():
    raise FileNotFoundError(f"入力ファイルが見つかりません: {input_tiff_path}")

raw = tifffile.imread(input_tiff_path)

expected_dtypes = {np.dtype("uint8"), np.dtype("uint16"), np.dtype("float32"), np.dtype("float64")}
dtype_status = "想定内" if raw.dtype in expected_dtypes else "想定外"

if raw.ndim == 2:
    image_2d = raw
elif raw.ndim == 3 and use_channel_selection:
    if raw.shape[-1] in (2, 3, 4):
        if not (0 <= channel_index < raw.shape[-1]):
            raise ValueError(f"channel_index は 0 以上 {raw.shape[-1] - 1} 以下にしてください。")
        image_2d = raw[..., channel_index]
    else:
        if not (0 <= channel_index < raw.shape[0]):
            raise ValueError(f"channel_index は 0 以上 {raw.shape[0] - 1} 以下にしてください。")
        image_2d = raw[channel_index]
else:
    raise ValueError(
        "この notebook は 2D TIFF 用です。raw.ndim == 2 の画像を指定してください。"
        "multi-channel TIFF の場合は use_channel_selection=True にしてチャンネルを選択してください。"
    )

image_2d = np.asarray(image_2d)

pd.DataFrame([
    {
        "項目": "元画像",
        "shape": str(raw.shape),
        "dtype": str(raw.dtype),
        "bit_depth": raw.dtype.itemsize * 8,
        "dtype_status": dtype_status,
        "min": float(np.nanmin(raw)),
        "max": float(np.nanmax(raw)),
    },
    {
        "項目": "使用画像",
        "shape": str(image_2d.shape),
        "dtype": str(image_2d.dtype),
        "bit_depth": image_2d.dtype.itemsize * 8,
        "dtype_status": dtype_status,
        "min": float(np.nanmin(image_2d)),
        "max": float(np.nanmax(image_2d)),
    },
])

## 6. 入力画像を表示

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(image_2d, cmap="gray")
plt.title("Input image")
plt.axis("off")
plt.show()

## 7. intensity を正規化

StarDist の推論前に、画像の intensity を percentile ベースで正規化します。ビット深度が 8-bit でも 16-bit でも、ここでおおむね同じスケールに揃えます。

初期値は `1.0` percentile から `99.8` percentile です。背景や飽和が強い場合はこの値を調整してください。

In [ ]:
image_norm = normalize(image_2d, percentile_low, percentile_high)

plt.figure(figsize=(7, 7))
plt.imshow(image_norm, cmap="gray", vmin=0, vmax=1)
plt.title("Normalized image")
plt.axis("off")
plt.show()

## 8. StarDist 2D 学習済みモデルを読み込み

In [ ]:
model = StarDist2D.from_pretrained("2D_versatile_fluo")

## 9. セグメンテーションを実行

`n_tiles_y` と `n_tiles_x` は画像を分割して推論する設定です。メモリ不足になる場合は `2` や `4` に増やしてください。

`use_custom_thresholds` はこの notebook 内の切り替え用変数です。`False` の場合は `predict_instances()` に `prob_thresh` と `nms_thresh` を渡さず、モデル側の既定値を使います。過検出や検出漏れがある場合だけ `True` にして、`prob_thresh` と `nms_thresh` を `predict_instances()` に渡します。

In [ ]:
n_tiles = (max(1, int(n_tiles_y)), max(1, int(n_tiles_x)))

predict_kwargs = {"n_tiles": n_tiles}
if use_custom_thresholds:
    predict_kwargs.update({"prob_thresh": prob_thresh, "nms_thresh": nms_thresh})

labels, details = model.predict_instances(image_norm, **predict_kwargs)

pd.DataFrame([
    {
        "検出オブジェクト数": int(labels.max()),
        "label_shape": str(labels.shape),
        "label_dtype": str(labels.dtype),
    }
])

## 10. 結果を表示

左に入力画像、右にラベルを重ねた画像を表示します。

In [ ]:
overlay = render_label(labels, img=image_2d)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(image_2d, cmap="gray")
axes[0].set_title("Input image")
axes[0].axis("off")

axes[1].imshow(overlay)
axes[1].set_title("StarDist overlay")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 11. 結果を Google Drive に保存

インスタンスごとに番号が分かれたラベル TIFF、全インスタンスを 1 つにまとめた二値マスク TIFF、確認用のオーバーレイ PNG、各インスタンスの面積・重心 CSV を保存します。CSV は Excel で文字化けしにくい UTF-8 BOM 付きで保存します。

In [ ]:
base_name = input_tiff_path.stem
label_path = output_dir / f"{base_name}_stardist_labels.tif"
binary_mask_path = output_dir / f"{base_name}_stardist_binary_mask.tif"
overlay_path = output_dir / f"{base_name}_stardist_overlay.png"
csv_path = output_dir / f"{base_name}_stardist_measurements.csv"

tifffile.imwrite(label_path, labels.astype(np.uint16 if labels.max() <= np.iinfo(np.uint16).max else np.uint32))
binary_mask = (labels > 0).astype(np.uint8) * 255
tifffile.imwrite(binary_mask_path, binary_mask)

plt.figure(figsize=(8, 8))
plt.imshow(overlay)
plt.axis("off")
plt.tight_layout(pad=0)
plt.savefig(overlay_path, dpi=200, bbox_inches="tight", pad_inches=0)
plt.close()

rows = []
for label_id in range(1, int(labels.max()) + 1):
    ys, xs = np.where(labels == label_id)
    if ys.size == 0:
        continue
    rows.append({
        "label_id": label_id,
        "area_px": int(ys.size),
        "centroid_y": float(ys.mean()),
        "centroid_x": float(xs.mean()),
    })

measurements = pd.DataFrame(rows)
measurements.to_csv(csv_path, index=False, encoding="utf-8-sig")

saved_files = [
    ("ラベル TIFF", label_path),
    ("二値マスク TIFF", binary_mask_path),
    ("オーバーレイ PNG", overlay_path),
    ("測定 CSV", csv_path),
]

display(Markdown("保存したファイル:\n" + "\n".join(
    f"- **{name}**  \n  `{path}`" for name, path in saved_files
)))

## 12. よくある調整点

- **ファイルが見つからない**: `input_tiff_path` が `/content/drive/MyDrive/...` から始まる Google Drive 上の正しいパスか確認してください。
- **3D TIFF と判定される**: この notebook は 2D 用です。3D stack は Fiji や Python で 2D slice に分けるか、slice-wise 処理用 notebook に変更してください。
- **RGB 画像だった**: `2D_versatile_fluo` は単一チャンネル蛍光画像向けです。蛍光チャンネルを選択するか、H&E 画像なら `2D_versatile_he` を検討してください。
- **検出漏れが多い**: `use_custom_thresholds=True` にして `prob_thresh` を下げる、または正規化 percentile を調整してください。
- **過検出が多い**: `prob_thresh` を上げる、`nms_thresh` を調整する、または画像の pixel size と対象物サイズがモデルの想定から大きく外れていないか確認してください。
- **メモリ不足**: `n_tiles_y` と `n_tiles_x` を `2` または `4` に増やしてください。
- **ビット深度について**: `uint8` と `uint16` はどちらも正規化してから推論するため使用可能です。重要なのは、対象物が背景から識別できる強度情報を持っていること、飽和や強いノイズが少ないことです。

## 参考情報

- StarDist GitHub: https://github.com/stardist/stardist
- StarDist FAQ: https://stardist.net/faq/
- ZeroCostDL4Mic: https://github.com/HenriquesLab/ZeroCostDL4Mic

## 引用

この notebook を研究・発表・論文で使用する場合は、StarDist 2D の原著論文と公式リポジトリを引用してください。

- Schmidt U, Weigert M, Broaddus C, Myers G. Cell Detection with Star-Convex Polygons. MICCAI 2018. DOI: https://doi.org/10.1007/978-3-030-00934-2_30
- StarDist 公式リポジトリ: https://github.com/stardist/stardist
- StarDist 公式 citation 情報: https://stardist.net/
